# cudf-bench: clock-lock verdict (Step 3e)

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. ~8 min. **Click Allow on the download at the end.**

Suspect: GPU clock boosting (DVFS) explains the call-to-call speed changes; a real ~1.4–1.7x cuDF skew penalty remains underneath. This run:

1. logs the **actual SM clock at every call** (correlation in black and white),
2. **locks the clocks** to a fixed frequency and reruns — under locked clocks the decay must vanish and gap/no-gap must match,
3. measures the clean skew penalty at fixed clocks (the number for the GitHub issue),
4. adds the missing uniform+gap control.

In [ ]:
!nvidia-smi -L
%pip install -q nvidia-ml-py

In [ ]:
try:
    import cudf
except ImportError:
    %pip install -q cudf-cu12
    import cudf
print("cudf", cudf.__version__)

In [ ]:
%cd /content
![ -d cudf-bench ] || git clone -q https://github.com/alexislowys/cudf-bench.git
%cd /content/cudf-bench
# VM clone is disposable — force it to match GitHub exactly, discard any leftovers
!git fetch -q && git reset -q --hard origin/main && git clean -qfd results
!rm -f results/transient3.csv
%pip install -q -e .

## Phase 1: free-running clocks (with per-call clock log)

In [ ]:
!python -m benchmark.transient --op groupby --skew 1.5 --str-cols 2 --iters 12
!python -m benchmark.transient --op groupby --skew 0   --str-cols 2 --iters 12
!python -m benchmark.transient --op groupby --skew 1.5 --str-cols 2 --iters 12 --gap 2
!python -m benchmark.transient --op groupby --skew 0   --str-cols 2 --iters 12 --gap 2

## Phase 2: lock clocks, rerun

`-lgc` pins SM clocks (T4 supports it; Colab runs as root). If the lock fails, phase 2 just duplicates phase 1 — still useful as a repeat.

In [ ]:
!nvidia-smi -lgc 1590,1590 || echo 'CLOCK LOCK FAILED - phase 2 = repeat of phase 1'
!nvidia-smi --query-gpu=clocks.sm,clocks.max.sm --format=csv

In [ ]:
!python -m benchmark.transient --op groupby --skew 1.5 --str-cols 2 --iters 12 --rmm-pool prealloc
!python -m benchmark.transient --op groupby --skew 0   --str-cols 2 --iters 12 --rmm-pool prealloc
!python -m benchmark.transient --op groupby --skew 1.5 --str-cols 2 --iters 12 --gap 2 --rmm-pool prealloc
!nvidia-smi -rgc || true

## Quick look

Note: phase 2 rows are tagged `rmm_pool=prealloc` (doubles as locked-clock marker and removes the allocator variable at once).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("results/transient3.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for (skew, gap, pool), g in df.groupby(["skew", "gap", "rmm_pool"]):
    lbl = f"skew={skew} gap={gap} {'LOCKED' if pool == 'prealloc' else 'free'}"
    axes[0].plot(g["iter"], g["wall_ms"], marker="o", label=lbl)
    axes[1].plot(g["iter"], g["sm_mhz"], marker="o", label=lbl)
axes[0].set_title("time per call (ms)"); axes[1].set_title("SM clock at call start (MHz)")
for ax in axes: ax.set_xlabel("call #"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## Auto-download (click Allow)

In [ ]:
from google.colab import files
files.download("/content/cudf-bench/results/transient3.csv")